# 第 18 课 · 本地流式服务与并发评估

模型在 Notebook 里答对，不代表服务交付正确。本课使用第 17 课合并模型，启动本机 vLLM OpenAI-compatible 接口，测量流式响应、并发、错误率与答案回归。无需外部 API 密钥，不上传数据，不开放互联网访问。

服务仅绑定 `127.0.0.1:18018`。端口被占用就停止，不连接或终止其他服务。本课启动的子进程在测试结束或异常时由 finally 清理。若强制杀死整个内核导致清理无法执行，可根据运行目录中的日志定位该进程；不要批量杀死其他 Ray/vLLM 作业。

## 1 · 固定评估协议

使用第 17 课的 24 条合成测试工单，temperature=0、max_tokens=12。加载 FP32 合并产物时显式转换为 BF16 以控制显存；这会引入额外数值差异，所以必须与 FP32 transformers 预测比较。并发为 1、2、4，每档运行三轮，轮换档位顺序；共 216 个计分请求，另有一个 warm-up 请求。重复请求用于测量服务行为，不增加独立语义样本数。

为减少重复前缀缓存造成的混淆，关闭 prefix cache；eager 模式与短上下文用于控制资源。这不是推荐的生产性能配置。模型加载与预热单独记录，不计入稳态吞吐。


In [ ]:
import importlib.util, json, time
from pathlib import Path
import pandas as pd
from IPython.display import display
def module(name,path):
    spec=importlib.util.spec_from_file_location(name,path)
    value=importlib.util.module_from_spec(spec);spec.loader.exec_module(value);return value
lesson=module('routing17',Path('advanced17/experiment.py'))
serving=module('serving18',Path('advanced18/serving.py'))
run=Path(Path('experiment17_output/latest_run.txt').read_text().strip())
assert (run/'merged/config.json').exists(), '请先完成第 17 课的模型导出'
splits=json.loads((run/'dataset.json').read_text())
rows=splits['test']
out=Path('experiment18_output');out.mkdir(exist_ok=True)
server_logs=run/f'serving-{time.time_ns()}'
print('服务模型:',run/'merged')
print('计分请求数:',len(rows)*3*3)


## 2 · 从流式 HTTP 响应测量延迟

TTFT 从客户端开始 HTTP 请求到收到第一个非空文本片段。空的角色声明不算首 token。端到端延迟到完整响应结束。每个请求保存 token usage、finish_reason、答案、成功状态和错误信息；失败请求留在分母中。

SSE 的一个 chunk 不一定等于一个 token。我们单独记录 chunk 数量与相邻 chunk 间隔，并用服务报告的 completion_tokens 估算 TPOT。由于 EOS 可能计入 token 数、多个 token 可能合并为一个 chunk，这个 TPOT 只是客户端近似值，不当作精确的逐 token 调度时间。

下面会启动并等待服务，完成压测后关闭。初始化可能需要几分钟，原始日志写入独立 Linux 运行目录。测量期间不要运行其他 GPU 实验。


In [ ]:
server=serving.LocalServer(run/'merged',server_logs)
started=time.perf_counter()
try:
    server.start()
    startup=time.perf_counter()-started
    print(f'本地服务已就绪，启动耗时 {startup:.1f}s。正在执行并发测试。',flush=True)
    summary=serving.benchmark(server,rows,lesson.messages,out,repeats=3)
    display(summary)
finally:
    server.stop()
    print('本课启动的服务已关闭。',flush=True)
(out/'startup.json').write_text(json.dumps(dict(startup_seconds=startup,server_stopped=True),indent=2))


## 3 · 回归检查：服务换了，答案是否也变了

将每次 HTTP 答案与同一合并权重在 transformers 中的答案比较。两种后端的算子与数值路径不同，temperature=0 也不保证逐位一致；发生分歧时应保存原始回答、tokenizer 和配置，检查模板及停止条件，而不是删除异常请求。

这里的“回归集”已经用于第 17 课测试，适合发现部署偏差，不能再宣称是从未查看的新泛化测试。


In [ ]:
requests_frame=pd.read_csv(out/'requests.csv',keep_default_na=False)
reference=pd.read_csv(run/'merged_predictions.csv',keep_default_na=False)[['id','answer']].rename(columns={'answer':'transformers_answer'})
paired=requests_frame.merge(reference,on='id',how='left',validate='many_to_one')
assert paired.transformers_answer.notna().all()
paired['same_answer']=paired.answer==paired.transformers_answer
report=dict(requests=len(paired),errors=int((~paired.success).sum()),
    answer_disagreements=int((~paired.same_answer).sum()),
    accuracy_including_errors=float(paired.correct.mean()),
    note='Repeated short synthetic prompts; not independent samples or a production SLA.')
(out/'regression.json').write_text(json.dumps(report,indent=2))
paired.to_csv(out/'regression_predictions.csv',index=False)
display(pd.DataFrame([report]))
display(paired[~paired.same_answer][['id','answer','transformers_answer','error']].head(10))


In [ ]:
import matplotlib.pyplot as plt
summary=pd.read_csv(out/'summary.csv')
fig,axes=plt.subplots(1,2,figsize=(10,3))
axes[0].plot(summary.concurrency,summary.output_tokens_per_second,marker='o')
axes[0].set(xlabel='Concurrency',ylabel='Output tokens / second',title='Throughput')
axes[1].plot(summary.concurrency,summary.e2e_p95_ms,marker='o',label='End-to-end p95')
axes[1].plot(summary.concurrency,summary.ttft_p95_ms,marker='o',label='TTFT p95')
axes[1].set(xlabel='Concurrency',ylabel='Milliseconds',title='Latency cost');axes[1].legend()
plt.tight_layout();plt.show();plt.close(fig)


## 4 · 怎样读懂 p95 与吞吐

本客户端采用闭环固定并发：最多 C 个 HTTP 请求同时执行，完成后才补下一个请求。每个请求的延迟从工作线程开始发请求计时，不包含它在客户端线程池排队的时间；吞吐的墙钟窗口包含整个批次。

因此，这个实验不能反映固定外部到达率下无限增长的排队延迟。生产压测还需要开环到达模型、长短请求混合、超时、取消、错误注入与持续运行。每档只有 72 个请求，p95 是描述性统计，不是稳定 SLA 保证。

吞吐定义为服务报告的输出 token 总数除以该档各轮测量窗口总时间，不是每请求 token/s 的简单平均。若提高并发后吞吐上升但 p95 变差，需要按业务延迟目标选配置，不能只追求峰值吞吐。

## 5 · 交付检查与扩展作业

把 base revision、adapter 配置、合并模型、tokenizer、chat template、解码参数、后端版本作为一组版本化产物。发布前固定回归集、失败率与回滚阈值；本课只运行本地临时服务，不是公网部署。

下一次实验只改变一个因素：打开 prefix cache、增加输出长度或提高并发。预先记录预期影响，再测质量与延迟。测试多卡、张量并行或量化时，需要重新验证数值与服务行为。

参考：[vLLM OpenAI-compatible server](https://docs.vllm.ai/en/latest/serving/openai_compatible_server/)。所有性能结论都应注明硬件、模型、请求分布和测量窗口。
